In [ ]:
# Install dependencies if running in a new environment
!pip install chromadb pypdf google-genai python-dotenv

import os
import chromadb
from pypdf import PdfReader
from google import genai
import dotenv

# Load environment variables
dotenv.load_dotenv()

In [ ]:
# Configuration Variables and Client Initialization
# Ensure the .env file contains GEMINI_API_KEY or replace the placeholder below
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Initialize the generative AI client
client = genai.Client()

# System Settings
PDF_FILE_PATH = "YOUR_PDF_FILE_PATH_HERE.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-2.5-flash"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

In [ ]:
def get_embedding(text: str) -> list[float]:
    """Generates a vector embedding for a given text string."""
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,  
        contents=text
    )
    return response.embeddings[0].values

def extract_and_chunk_pdf(pdf_path: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """Reads a PDF and breaks it into overlapping chunks of text to preserve semantic context."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"The specified file was not found: {pdf_path}")
        
    reader = PdfReader(pdf_path)
    full_text = ""
    
    # Extract text from all pages
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
            
    # Apply overlapping chunking strategy
    chunks = []
    start = 0
    while start < len(full_text):
        end = start + chunk_size
        chunks.append(full_text[start:end])
        start += (chunk_size - overlap)
        
    return chunks

In [ ]:
# Process the document and initialize the vector database
print("Starting document processing...")

chunks = extract_and_chunk_pdf(PDF_FILE_PATH, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
print(f"Document split into {len(chunks)} chunks.")

# Initialize ChromaDB persistent client
# Use get_or_create_collection to prevent errors on subsequent executions
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="document_collection")

# Generate embeddings and populate the database
print("Generating embeddings and populating database...")
for i, chunk in enumerate(chunks):
    embedding = get_embedding(chunk)
    unique_id = f"{os.path.basename(PDF_FILE_PATH)}_chunk_{i}"
    
    collection.upsert(
        ids=[unique_id],
        embeddings=[embedding],
        documents=[chunk]
    )
    
print("Vector database populated successfully.")

In [ ]:
def query_rag_system(query: str, n_results: int = 3) -> str:
    """Processes a query, retrieves context from ChromaDB, and generates an answer."""
    
    query_vector = get_embedding(query)
    
    # Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results
    )
    
    retrieved_chunks = results.get("documents", [[]])[0]
    context = "\n\n---\n\n".join(retrieved_chunks)
    
    # Construct strictly grounded prompt
    prompt = (
        f"Answer the user's question using ONLY the context provided below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    ) 
    
    # Generate content
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
    )
    
    return response.text

# Execution loop for notebook environments
print("System ready. Enter queries below (type 'exit' to terminate).")
while True:
    user_query = input("Query: ").strip()
    
    if user_query.lower() == 'exit':
        print("Exiting...")
        break
    if not user_query:
        continue
        
    answer = query_rag_system(user_query)
    
    print(f"\nResponse: {answer}\n")
    print("-" * 60)